In [1]:
import weaviate
from weaviate.classes.config  import Configure,Property, DataType
import requests, json
import base64
from pathlib import Path
import os


In [2]:

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)

In [3]:
collections = client.collections.delete("Grounded_nomic_full")
print(collections)

None


In [4]:

questions = client.collections.create(
    name="Grounded_nomic_full",
    vector_config=Configure.Vectors.text2vec_ollama(  # Configure the Ollama embedding integration
        api_endpoint="http://172.17.0.3:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="nomic-embed-text",  # The model to use
    ),
    generative_config=Configure.Generative.ollama(  # Configure the Ollama generative integration
        api_endpoint="http://172.17.0.3:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="llama3.2",  # The model to use
    ),
    properties=[
        Property(
            name="type",
            data_type=DataType.TEXT,
            skip_vectorization=True
        ),
        Property(
            name="page",
            data_type=DataType.INT,
            skip_vectorization=True
        ),
        Property(
            name="description",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="text",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="trace",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="filename",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="image",
            data_type=DataType.BLOB,
            skip_vectorization=True,  # Image won't be embedded
            index_null_state=True
        ),
    ]
)


In [5]:
questions = client.collections.use("Grounded_nomic_full")


In [7]:
for filename in sorted(os.listdir("../clean_chunks")):
    if filename.endswith(".json"):
        filepath = os.path.join("../clean_chunks", filename)
            
    with open(filepath, 'r') as f:
        data = json.load(f)
    print(f"Importing data from {filename} with {len(data)} entries...")
    with questions.batch.fixed_size(batch_size=200) as batch:
        for d in data:
            properties = {
                    "type": d["block_type"],
                    "page": d["page"],
                    "description": d["description"],
                    "text": d["text"],
                    "trace": d["trace"],
                    "filename": d["filename"],
                #    "Image": list(d["images"].values())[0] if d["images"] else None,
                }

            # Handle image properly
            if d["images"]:
                properties["image"] = d["images"]
        
            batch.add_object(properties)
            
            if batch.number_errors > 10:
                print("Batch import stopped due to excessive errors.")
                break

    failed_objects = questions.batch.failed_objects
    if failed_objects:
        print(f"Number of failed imports: {len(failed_objects)}")
        print(f"failed on flename:{filepath}")
        print(f"First failed object: {failed_objects[0]}")
        
    print(f"Finished importing data from {filename}")

client.close()  # Free up resources

Importing data from O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json with 627 entries...
Finished importing data from O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json
Importing data from O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json with 384 entries...
Finished importing data from O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json
Importing data from O-RAN.SFG.Non-RT-RIC-Security-TR-v01.00_cleaned.json with 413 entries...
Finished importing data from O-RAN.SFG.Non-RT-RIC-Security-TR-v01.00_cleaned.json
Importing data from O-RAN.SuFG.CE-v01.00_cleaned.json with 181 entries...
Finished importing data from O-RAN.SuFG.CE-v01.00_cleaned.json
Importing data from O-RAN.SuFG.TR.NES-Analysis-R004-v01.01_cleaned.json with 339 entries...
Finished importing data from O-RAN.SuFG.TR.NES-Analysis-R004-v01.01_cleaned.json
Importing data from O-RAN.TIFG.CGofOTIC.0-v06.00_cleaned.json with 279 entries...
Finished importing data from O-RAN.TIFG.CGofOTIC.0-v06.00_cleaned.json
Importing data from O-RAN.TIFG.E

In [8]:
import weaviate
import json
from weaviate.classes.query import Filter
client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)
questions = client.collections.use("Grounded_nomic_full")

my_filter = Filter.by_property("image").is_none(False)


response = questions.query.bm25(

    query=""" 

   The diagram shows interactions between Near-RT RICs and E2 Nodes:

"RIC SUBSCRIPTION MODIFICATION REQUIRED" from Near-RT RIC to E2 Node
"RIC SUBSCRIPTION MODIFICATION REFUSE" from E2 Node to Near-RT RIC
        """,
    limit=10
    
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=2))

client.close()  # Free up resources

{
  "type": "FigureGroup",
  "trace": "8.2 RIC Functional procedures --> 8.2.6 RIC Subscription Modification Required procedure --> 8.2.6.1 General --> Figure 8.2.6.3-1: RIC Subscription Modification Required procedure, unsuccessful operation",
  "description": "The diagram shows interactions between a Near-RT RIC and an E2 Node:\n1. \"RIC SUBSCRIPTION MODIFICATION REFUSE\" from Near-RT RIC to E2 Node\n2. \"RIC SUBSCRIPTION MODIFICATION REQUIRED\" from E2 Node to Near-RT RIC",
  "filename": "O-RAN.WG3.TS.E2AP-R004-v08.00",
  "text": "Figure 8.2.6.3-1: RIC Subscription Modification Required procedure, unsuccessful operation",
  "page": 26
}
{
  "type": "FigureGroup",
  "trace": "8.2 RIC Functional procedures --> 8.2.6 RIC Subscription Modification Required procedure --> 8.2.6.1 General --> Figure 8.2.6.2-1: RIC Subscription Modification Required procedure, successful operation",
  "description": "The diagram shows interactions between a Near-RT RIC and an E2 Node:\n1. \"RIC SUBSCRIPTION

In [9]:
client.close()